# Etapa 0: ingesta y buffer

Este notebook importa los módulos del proyecto y consulta resultados persistidos. Primero ejecuta los comandos de preparación del README. Solo demuestra la etapa 0.

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd()
if not (ROOT / "Plan_proyecto.md").exists():
    ROOT = ROOT.parent
assert (ROOT / "Plan_proyecto.md").exists(), "Ejecuta desde la raíz o notebooks/"
sys.path.insert(0, str(ROOT))
from src.pipeline.stage0 import run_stage0
config = json.loads((ROOT / "configs/stage0.json").read_text())


## Composición y reglas de preparación

Cada dataset tiene 13 eventos reservados; los restantes son filas del prefijo histórico. Las órdenes de compras reales son agrupaciones sintéticas y pueden quedar cortadas al final del prefijo.

In [ ]:
for version in ("1K", "10K"):
    manifest = json.loads((ROOT / "data/prepared" / version / "manifest.json").read_text())
    print(version, manifest["counts"], manifest["validation"])


## Verificación persistida

Compara contenido, identidad y orden de todos los eventos de entrada y salida, además de los hashes y las entidades. Son corridas de verificación, sin repeticiones estadísticas.

In [ ]:
report = json.loads((ROOT / "results/verification.json").read_text())
print("Verificación:", report["passed"])
for replay in report["replays"]:
    print(replay["mode"], "eventos:", replay["verified_events"],
          "pico FIFO:", replay["queue_peak"], "capacidad:", replay["queue_capacity"],
          "bloqueos:", replay["blocked_puts"])


## Ejecución local opcional

La salida usa un nombre nuevo en cada corrida. El sobre conserva el evento y añade referencias a entidades y el resultado de la etapa 0, disponible para la futura etapa 1.

In [ ]:
from uuid import uuid4
output = ROOT / "results" / f"notebook_{uuid4().hex}.jsonl"
stats = await run_stage0(ROOT / "data/prepared/1K/events.jsonl", output, config, mode="unlimited")
print(stats)
with output.open() as stream:
    envelope = json.loads(next(stream))
envelope
